In [0]:
spark.version


In [0]:
%run ../tests/test_silver_layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import logging

In [0]:
# Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("Holidays Silver Layer")

**Databricks Widgets Set Up**

In [0]:
dbutils.widgets.text("bronze_holidays_table", "")
dbutils.widgets.text("silver_holidays", "")

bronze_holidays_table = dbutils.widgets.get("bronze_holidays_table") or "chicago_taxi_data.bronze.bronze_holidays"
silver_holidays_data = dbutils.widgets.get("silver_holidays") or "chicago_taxi_data.silver.silver_holidays"


In [0]:
df_bronze_holidays = spark.read.format("delta").table(bronze_holidays_table)

In [0]:
display(df_bronze_holidays.limit(5))

In [0]:
df_bronze_holidays.printSchema()

**Type Cast And Select Required Columns**

In [0]:
df_silver_holidays = df_bronze_holidays.select(
    F.to_date("date").alias("date"),
    F.col("holiday_name").alias("holiday")
    )

**Check Distinct Holidays**

In [0]:
df_silver_holidays.select(F.col("holiday")).show()

**Create Dataframe With Additional Holidays**

In [0]:
manual_data = [
    ("2025-02-14", "Valentine's Day"),
    ("2025-03-17", "St. Patrick's Day"),
    ("2025-04-20", "Easter Sunday"),
    ("2025-10-31", "Halloween"),
    ("2025-11-28", "Black Friday"),
    ("2025-12-24", "Christmas Eve"),
    ("2025-12-31", "New Year's Eve"),
    ("2026-02-14", "Valentine's Day"),
    ("2026-03-17", "St. Patrick's Day"),
    ("2026-04-05", "Easter Sunday"),
    ("2026-10-31", "Halloween"),
    ("2026-11-27", "Black Friday"),
    ("2026-12-24", "Christmas Eve"),
    ("2026-12-31", "New Year's Eve")
]

# 3. Create a small DataFrame from these manual dates
df_manual = spark.createDataFrame(manual_data, ["date", "holiday"])
df_manual = df_manual.withColumn("date", F.to_date("date"))

In [0]:
display(df_manual)

**Union With Existent Holidays Data**

In [0]:
df_silver_holidays = df_silver_holidays.unionByName(df_manual).dropDuplicates(["date"])

In [0]:
display(df_silver_holidays)

**Data Quality Check**

In [0]:

required_columns = ["date", "holiday"]
validate_schema(df_silver_holidays, required_columns)

try:
    logger.info("Starting DQ checks Silver Layer")
    
    validate_no_nulls(df_silver_holidays, "date")
    validate_schema(df_silver_holidays, required_columns)
    validate_duplicates(df_silver_holidays, ["date"])

    
    logger.info("Silver Holidays DQ tests Passed.")
except Exception as e:
    logger.error(f"DQ Failed: {str(e)}")
    dbutils.notebook.exit(str(e))

In [0]:
try:
    logger.info(f"Writing df_silver_holidays to Silver Layer")
    df_silver_holidays.write.format("delta").mode("overwrite").saveAsTable(silver_holidays_data)
    logger.info(f"df_silver_holidays saved successfully to `{silver_holidays_data}` Delta Table")
except Exception as e:
    logger.error(f"Error writing df_silver_holidays to Silver Layer: {e}")
    dbutils.notebook.exit(e)
